# Baseline с masked loss + packing — Шаги 1 и 2 (быстрая версия с упаковкой)

Дообучение базовой модели `Qwen2.5-1.5B` на случайной подвыборке `d0rj/ru-instruct` (90 000 примеров, 1 эпоха), loss считается **только по токенам ответа ассистента**, и для ускорения короткие примеры **упаковываются** в одну последовательность до `max_length=2048`.

**Разница с `baseline_masked.ipynb` — ровно один параметр**: `training_args.packing = True`. Всё остальное (data prep, маскирование через `train_on_responses_only`, гиперпараметры) — идентично.

## Почему именно так, а не `assistant_only_loss=True`

В Unsloth 2026.5.x compiled-обёртка `UnslothSFTTrainer._prepare_dataset` **отказывается** обрабатывать conversational dataset (поле `messages`) без `formatting_func`:

```
RuntimeError: Unsloth: You must specify a `formatting_func`
```

Это барьер **до** TRL — никакие `packing=True` или `assistant_only_loss=True` не спасают, потому что мы даже до создания `SFTTrainer` не доходим. Если дать `formatting_func`, рендерящий messages в текст, то TRL теряет структуру и `assistant_only_loss` не может найти assistant-токены.

**Выход:** pre-render text заранее (как в `baseline_masked.ipynb`), маскировать через `train_on_responses_only` после создания тренера. В свежих версиях Unsloth (см. discussion #2828) `train_on_responses_only` корректно обрабатывает **множественные пары маркеров** в одной упакованной последовательности — это и нужно для packing.

## Что меняется в `SFTConfig`

- **`packing=True`** — короткие примеры склеиваются в одну последовательность длиной до `MAX_SEQ_LEN`. Padding почти исчезает. На `d0rj/ru-instruct` (длины 50–2000+) даёт ~1.5–2× ускорение wall-time на L4.
- **`group_by_length` снят** — с packing он не нужен.
- **`gradient_checkpointing=True` убран из `SFTConfig`** — Unsloth уже включил его в `get_peft_model(use_gradient_checkpointing="unsloth")`.

## Методологическая оговорка vs `baseline_full`

- `baseline_full` обучен **без packing**. Сравнение `Δppl(full − masked_packed)` смешивает два эффекта: (1) маскирование prompt-токенов и (2) packing. Чтобы получить чистый ablation эффекта **только маскирования**, сравнивай с `baseline_masked.ipynb` (тоже без packing). Чтобы выделить эффект **только packing**, сравнивай `baseline_masked` ↔ `baseline_masked_packed`.

## Почему 1 эпоха на 90k, а не 3 эпохи на 30k

Старый режим `30k × 3 эпохи` давал примерно те же 85.5k train-примеров, увиденных оптимизатором, но только 28.5k уникальных train-примеров. Так как полный `d0rj/ru-instruct` сильно больше, методологически лучше пройти 1 раз по 90k случайных примеров: compute примерно тот же, уникальных данных в 3 раза больше, риск переобучения на маленькую случайную подвыборку ниже.

**Ожидаемое время на L4:** примерно как у старого packed-запуска на `30k × 3`, потому что число optimizer steps остаётся близким.

## Шаг 1.1 — Установка зависимостей

Чистая переустановка ключевых пакетов + установка свежего `unsloth` с PyPI (он сам подберёт совместимые `transformers`, `datasets`, `trl`, `peft`, `torchao`). Жёстко пинить версии не пытаемся: сборки Colab меняются, Unsloth выкатывает новый релиз каждый месяц, и попытка зафиксировать всё ломается на следующей неделе.

После установки **обязательно** перезапустить runtime (Colab поднимет уведомление в конце ячейки). После рестарта продолжаем со следующей ячейки.

In [ ]:
# Сносим всё, что мы могли поставить кривой версии в предыдущих попытках.
# Ошибки игнорируем — пакета может и не быть.
!pip uninstall -y -q \
    transformers datasets trl peft \
    unsloth unsloth_zoo torchao \
    2>/dev/null

# Ставим unsloth с PyPI — он сам подтянет совместимые transformers/datasets/trl/peft/torchao.
!pip install --quiet --upgrade unsloth

# Дополняем тем, что unsloth не тянет, но нам нужно для SFT и квантизации.
!pip install --quiet --upgrade bitsandbytes accelerate sentencepiece

## Шаг 1.1.5 — Проверка версий

Печатаем фактические версии установленных пакетов. От этого зависит API SFTConfig (в `trl>=0.20` `max_seq_length` переименован в `max_length`, а `dataset_text_field` депрекейтнут), а также то, какие функции Unsloth доступны.

Запускать **после** рестарта runtime, прежде чем идти дальше.

In [ ]:
import importlib.metadata as _md

print("Установленные версии:")
for pkg in [
    "unsloth", "unsloth_zoo",
    "transformers", "trl", "peft", "datasets",
    "torchao", "bitsandbytes", "accelerate",
    "torch",
]:
    try:
        v = _md.version(pkg)
    except _md.PackageNotFoundError:
        v = "MISSING"
    print(f"  {pkg:15s} {v}")

## Шаг 1.2 — Проверка железа и обязательный bf16

T4 (Turing, sm_75) **не поддерживает bf16**. A100/L4 (Ampere+, sm_80+) — поддерживает. Если bf16 недоступен, ноутбук должен остановиться: fallback на fp16 в этом эксперименте запрещён.

Не используем `torch.cuda.is_bf16_supported()`: в PyTorch есть [баг](https://github.com/pytorch/pytorch/issues/118122), из-за которого функция возвращает `True` для Turing. Проверяем напрямую через compute capability.

> **Важно:** `import unsloth` обязательно делаем здесь, до любого импорта из `transformers`. Иначе теряются оптимизации Unsloth (FastAttention, патчи torch и т.д.) и появляется warning про порядок импортов.

In [ ]:
import unsloth   # ОБЯЗАТЕЛЬНО до transformers
import torch


def detect_precision():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA недоступна. Для этого запуска нужен GPU с bf16 (A100/L4/H100 или другой sm_80+).")
    major, minor = torch.cuda.get_device_capability(0)
    if major < 8:
        raise RuntimeError(
            f"bf16 недоступен на GPU sm_{major}{minor}. "
            "Fallback на fp16 отключён намеренно; запусти на A100/L4/H100 или другом sm_80+."
        )
    return {"bf16": True, "fp16": False}


def check_hardware():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA недоступна. Для этого запуска нужен GPU с bf16 (A100/L4/H100 или другой sm_80+).")

    name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    major, minor = torch.cuda.get_device_capability(0)
    p = detect_precision()

    print(f"GPU:                {name}")
    print(f"VRAM:               {vram_gb:.1f} GB")
    print(f"Compute capability: sm_{major}{minor}")
    print("Precision:          bf16 (required)")

    if vram_gb < 4:
        print("СОВЕТ: VRAM < 4 GB — переключитесь на Qwen2.5-0.5B")
    elif vram_gb < 8:
        print("OK: Qwen2.5-1.5B + 4bit поместится с запасом")
    else:
        print("OK: можно использовать Qwen2.5-1.5B или 3B")


check_hardware()

## Шаг 1.2.5 — Google Drive (для устойчивости к падению сессии)

Чекпоинты копятся каждые 200 шагов. Если сессия Colab упадёт (а она упадёт рано или поздно — таймаут, разрыв сети, OOM), всё в `/content/` пропадает. Чтобы не терять часы обучения, монтируем Google Drive и сохраняем туда.

После монтирования: даже после полного дисконнекта можно заново открыть ноутбук, прогнать все ячейки сверху вниз — `trainer.train(resume_from_checkpoint=True)` подхватит с последнего чекпоинта.

Если запускаешь локально (не Colab) или не хочешь Drive — поставь `USE_DRIVE = False`, сохранение пойдёт в текущую папку.

Для параллельных запусков: если `/content/drive` уже занят локальными файлами или другой тетрадкой, ячейка пробует запасные mountpoint-ы (`/content/gdrive`, `/content/drive_1`, ...). `force_remount=True` намеренно не используется, чтобы не сломать соседний прогон.


In [ ]:
USE_DRIVE = True   # False = сохранять локально (потеряется при разрыве сессии)

if USE_DRIVE:
    try:
        import os
        from google.colab import drive

        def _is_drive_mounted(path):
            return os.path.isdir(os.path.join(path, "MyDrive"))

        def _mount_drive_parallel_safe():
            # Не используем force_remount=True: это может отвалить Drive у другой тетрадки.
            candidates = ["/content/drive", "/content/gdrive"] + [
                f"/content/drive_{i}" for i in range(1, 8)
            ]

            for mountpoint in candidates:
                if _is_drive_mounted(mountpoint):
                    print(f"Google Drive уже доступен: {mountpoint}")
                    return mountpoint

            last_error = None
            for mountpoint in candidates:
                try:
                    if os.path.exists(mountpoint):
                        if not os.path.isdir(mountpoint):
                            print(f"{mountpoint} занят файлом — пробуем следующий")
                            continue
                        if os.listdir(mountpoint):
                            print(f"{mountpoint} занят локальными файлами — пробуем следующий")
                            continue

                    os.makedirs(mountpoint, exist_ok=True)
                    drive.mount(mountpoint)
                    if _is_drive_mounted(mountpoint):
                        return mountpoint
                    print(f"WARNING: {mountpoint} смонтирован, но MyDrive не найден; используем как есть")
                    return mountpoint
                except ValueError as exc:
                    last_error = exc
                    if _is_drive_mounted(mountpoint):
                        return mountpoint
                    print(f"{mountpoint}: {exc}")

            raise RuntimeError(
                "Не удалось подключить Google Drive: все mountpoint заняты. "
                "Очисти локальные /content/drive* или поставь USE_DRIVE=False."
            ) from last_error

        DRIVE_MOUNT = _mount_drive_parallel_safe()
        OUTPUT_BASE = f"{DRIVE_MOUNT}/MyDrive/diploma"
        print(f"OUTPUT_BASE = {OUTPUT_BASE} (Google Drive)")
    except ImportError:
        print("google.colab недоступен — сохраняем локально")
        OUTPUT_BASE = "."
else:
    OUTPUT_BASE = "."
    print(f"OUTPUT_BASE = {OUTPUT_BASE} (локально)")


## Шаг 1.3 — Конфиг

Все константы в одном месте, чтобы потом для других стратегий отбора менялся только `OUTPUT_DIR` и логика формирования `train_dataset`.

`SUBSAMPLE_SIZE = 90_000`, `NUM_EPOCHS = 1` — это обновлённый бюджет baseline. Он заменяет старый режим `30_000 × 3 эпохи`: число увиденных train-примеров остаётся примерно тем же, но вместо трёх повторов маленькой подвыборки модель видит втрое больше уникальных инструкций.

In [ ]:
MODEL_NAME      = "Qwen/Qwen2.5-1.5B"
DATASET_NAME    = "d0rj/ru-instruct"

SUBSAMPLE_SIZE  = 90_000

MAX_SEQ_LEN     = 2048              # L4 24GB — обрезаем меньше длинных примеров
BATCH_SIZE      = 4                 # на ru-instruct batch=8 даёт больше padding waste, чем 4×2
GRAD_ACCUM      = 2                 # эффективный batch = BATCH_SIZE * GRAD_ACCUM = 8 (как в baseline_full)
LEARNING_RATE   = 2e-4
NUM_EPOCHS      = 1
WARMUP_RATIO    = 0.03
SEED            = 42
VAL_SIZE        = 0.05
COMMON_VAL_HOLDOUT_SIZE = int(SUBSAMPLE_SIZE * VAL_SIZE)  # внешний common val, исключён из train/selection pools

OUTPUT_DIR      = f"{OUTPUT_BASE}/outputs/baseline_masked_packed_base_90k_1ep_noleak"
ADAPTER_DIR     = f"{OUTPUT_DIR}/adapter"
LOG_PATH        = f"{OUTPUT_DIR}/training_log.json"
METRICS_PATH    = f"{OUTPUT_DIR}/final_metrics.json"

EVAL_SAMPLES        = 500
FINAL_EVAL_SAMPLES  = 200

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

## Шаг 1.3.5 — Опционально: удалить артефакты прошлого прогона

В следующей ячейке **`RESET_PREVIOUS_RUN = True`** полностью удаляет **`OUTPUT_DIR`** (все `checkpoint-*`, `adapter/`, логи, `final_metrics.json`). Используй перед **чистым** перезапуском с тем же сидом, чтобы не сработал **resume** с чужим `global_step` и не смешались артефакты.

**Внимание:** на смонтированном Google Drive удаление необратимо.

> Нумерация ячеек ниже сдвинута на **+2** (бывшие Cell 12 / 16 / 22 в тексте → **14 / 18 / 24**).


In [ ]:
import os
import shutil

# True = полностью снести OUTPUT_DIR и создать пустую (чистый прогон, без resume).
RESET_PREVIOUS_RUN = False

if RESET_PREVIOUS_RUN:
    if os.path.isdir(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)
        print(f"Удалено: {OUTPUT_DIR}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print("Пустая OUTPUT_DIR создана.")
else:
    print("RESET_PREVIOUS_RUN=False — OUTPUT_DIR не трогаем.")


## Шаг 1.4 — Подготовка датасета

`d0rj/ru-instruct` отдаёт записи в формате:
```
{
  "conversations": [{"role": "system|user|assistant", "content": "..."}, ...],
  "source": "имя_исходного_датасета"
}
```

Рендерим ChatML-шаблон Qwen2.5 в поле `text` через `dataset.map(batched=True)`. У `Qwen/Qwen2.5-1.5B` есть стоковый chat template, но для воспроизводимости явно ставим Unsloth-шаблон `qwen-2.5`, чтобы маркеры `<|im_start|>user\n` и `<|im_start|>assistant\n` совпадали при подготовке данных, обучении и оценке. Это снимает все претензии Unsloth-обёртки к conversational формату (она требует либо `dataset_text_field`, либо `formatting_func`), и одновременно даёт packing-у текст, который можно склеивать. Маскирование prompt-токенов сделает `train_on_responses_only` уже после создания trainer-а — он умеет находить **множественные** пары маркеров в упакованной последовательности.

Колонку `length` **не** добавляем — с packing нет `group_by_length`, лишний расчёт не нужен.

**Чекпоинт после запуска:** печатается размер сабсэмпла, средняя/максимальная длина в токенах, % обрезаемых примеров. Если `pct_over_max_seq_len > 20%` — поднять `MAX_SEQ_LEN` до 2048.

In [ ]:
import json
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from transformers import AutoTokenizer


def load_and_prepare(verbose=True):
    ds_full = load_dataset(DATASET_NAME, split="train")
    if verbose:
        print(f"Полный размер: {len(ds_full)} примеров")

    train_start = COMMON_VAL_HOLDOUT_SIZE
    train_end = train_start + SUBSAMPLE_SIZE
    if train_end > len(ds_full):
        raise ValueError(
            f"COMMON_VAL_HOLDOUT_SIZE + SUBSAMPLE_SIZE = {train_end} больше размера датасета ({len(ds_full)})"
        )

    ds_shuffled = ds_full.shuffle(seed=SEED)
    # Внешний common val = ds_shuffled[:COMMON_VAL_HOLDOUT_SIZE].
    # Обучающий random baseline начинается строго после него, чтобы финальная метрика не утекала в train.
    ds = ds_shuffled.select(range(train_start, train_end))
    if verbose:
        print(f"Common val holdout: ds_shuffled[0:{COMMON_VAL_HOLDOUT_SIZE}] (не используется в train)")
        print(f"Сабсэмпл train pool: ds_shuffled[{train_start}:{train_end}] → {len(ds)} примеров")

    tok = get_chat_template(
        AutoTokenizer.from_pretrained(MODEL_NAME),
        chat_template="qwen-2.5",
    )

    # Рендерим ChatML-шаблон Qwen2.5 заранее — снимаем требование Unsloth-обёртки
    # к formatting_func для conversational-датасетов. Маскирование prompt-токенов
    # сделает train_on_responses_only ПОСЛЕ создания trainer-а.
    def render(examples):
        texts = [
            tok.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
            for c in examples["conversations"]
        ]
        return {"text": texts}

    ds = ds.map(render, batched=True, remove_columns=ds.column_names, desc="render chat template")

    ds_split = ds.train_test_split(test_size=VAL_SIZE, seed=SEED)
    if verbose:
        print(f"Train: {len(ds_split['train'])}, Val: {len(ds_split['test'])}")

    if verbose:
        sample_n = min(1000, len(ds_split["train"]))
        lengths = [
            len(tok(ex["text"], add_special_tokens=False)["input_ids"])
            for ex in ds_split["train"].select(range(sample_n))
        ]
        too_long = sum(1 for l in lengths if l > MAX_SEQ_LEN)
        stats = {
            "avg_len_tokens": round(sum(lengths) / len(lengths), 1),
            "max_len_tokens": max(lengths),
            "pct_over_max_seq_len": round(too_long / len(lengths) * 100, 2),
            "expected_pack_ratio": round(MAX_SEQ_LEN / (sum(lengths) / len(lengths)), 2),
        }
        print("Статистика длин (выборка 1000):")
        print(json.dumps(stats, indent=2, ensure_ascii=False))
        print(f"  → packing ожидает ~{stats['expected_pack_ratio']:.1f} примеров на одну упакованную последовательность")

    return ds_split, tok


ds_split, _tokenizer_preview = load_and_prepare()
print("\n--- Пример отрендеренного текста ---")
print(ds_split["train"][0]["text"][:500])

## Шаг 1.5 — Загрузка модели + LoRA

QLoRA: модель в 4-bit, LoRA-адаптер сверху. По плану:
- `r=16`, `lora_alpha=16`
- Целевые модули — все линейные слои attention и MLP
- `lora_dropout=0` обязательно при 4-bit
- `use_gradient_checkpointing="unsloth"` экономит ещё ~30% VRAM (Unsloth-овская реализация быстрее transformers-native)

После загрузки модели прогоняем `tokenizer` через `get_chat_template(..., chat_template="qwen-2.5")` — фиксируем тот же ChatML-шаблон Qwen2.5, который использовался в Cell 14. Это важно для `train_on_responses_only`: маркеры `<|im_start|>user\n` и `<|im_start|>assistant\n` должны совпадать с тем, что реально лежит в input_ids. Это не потому, что base-модель не поддерживает `apply_chat_template`, а чтобы не зависеть от возможных изменений tokenizer config / версии `transformers` между подготовкой данных и запуском обучения.

**Чекпоинт:** доля обучаемых параметров — ожидаем 1–2%.

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template


def load_model_with_lora():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
    )

    # Фиксируем Unsloth-овский ChatML-шаблон Qwen2.5 — гарантия согласованности
    # маркеров между prepare_data, тренировкой и инференсом
    tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Обучаемых параметров: {trainable:,} ({100 * trainable / total:.2f}%)")

    return model, tokenizer


model, tokenizer = load_model_with_lora()

## Шаг 2.1 — Обучение с `packing=True` + `train_on_responses_only`

Конфигурация:
- 1 эпоха на ~85 500 train-примерах из случайной подвыборки 90k, **эффективный batch = 8** (BATCH_SIZE=4 × GRAD_ACCUM=2).
- **MAX_SEQ_LEN = 2048**: обрезается ~2–3% длинных примеров.
- Датасет в текстовом формате (`text`, отрендерен в Cell 14) — Unsloth-обёртка пропускает.
- **`packing=True`** (ставится атрибутом на `training_args`, см. ниже). Короткие примеры склеиваются в одну последовательность до `MAX_SEQ_LEN`. С средней длиной ~600 токенов на `d0rj/ru-instruct` получаем ~3 примера в одну packed-последовательность. Новый режим `90k × 1` даёт примерно столько же optimizer steps, сколько старый `30k × 3`, но без трёх повторных проходов по одним и тем же данным.
- **`train_on_responses_only`** вызывается **после** `SFTTrainer(...)`. Helper переписывает `labels[i] = -100` для всех токенов до response-маркера `<|im_start|>assistant\n` — корректно обрабатывает несколько пар маркеров в одной упакованной последовательности (Unsloth discussion #2828).
- `warmup_ratio=0.03` вместо фиксированных `warmup_steps=500`: при изменении размера подвыборки и числа эпох warmup масштабируется от реального числа шагов, а не остаётся случайно завышенным.

**Разбор замечания про эпохи:** руководитель прав. Для случайного baseline на большом датасете `30k × 3` хуже, чем `90k × 1`: compute почти тот же, но в первом варианте модель трижды видит одну и ту же маленькую подвыборку. Это повышает риск переобучения и снижает покрытие данных. Поэтому здесь фиксируем `SUBSAMPLE_SIZE=90_000`, `NUM_EPOCHS=1`.

**Что НЕ делаем и почему:**
- **`assistant_only_loss=True`** — несовместимо с pre-rendered text. Этот флаг работает только когда датасет conversational (поле `messages`) и TRL сам применяет training chat template. С отрендеренным `text` структура потеряна. Поэтому маскируем через Unsloth-овский `train_on_responses_only`.
- **`gradient_checkpointing=True` в `SFTConfig`** — Unsloth уже включил его в `get_peft_model(use_gradient_checkpointing="unsloth")`. Дублирующее включение может конфликтовать.
- **`group_by_length=True`** — с packing бесполезен, все packed-последовательности и так длиной `MAX_SEQ_LEN`.

**Методологические риски vs `baseline_full` и `baseline_masked` (без packing):**
- `baseline_full` обучен **без** packing. Сравнение `Δppl(full − masked_packed)` смешивает эффекты маскирования и упаковки.
- Чтобы выделить **только** эффект packing — сравнивай `baseline_masked` ↔ `baseline_masked_packed` (оба с маскированием, разница только в packing).
- Чтобы выделить **только** эффект маскирования при packing — нужен ещё `baseline_full_packed` (full-loss + packing). Если ppl `baseline_masked_packed` окажется заметно меньше, имеет смысл потом сделать и его.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

precision = detect_precision()
print("Используем bf16")

eval_subset = ds_split["test"].select(range(min(EVAL_SAMPLES, len(ds_split["test"]))))
print(f"Train: {len(ds_split['train'])}, in-loop eval: {len(eval_subset)}")

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    # gradient_checkpointing уже включён через use_gradient_checkpointing="unsloth"
    # в FastLanguageModel.get_peft_model().
    optim="adamw_8bit",
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,

    bf16=precision["bf16"],
    fp16=precision["fp16"],

    max_length=MAX_SEQ_LEN,

    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=400,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",

    dataloader_num_workers=0,
    dataloader_pin_memory=True,

    seed=SEED,
    data_seed=SEED,
)

# packing ставим атрибутом на объекте, а не kwargs в SFTConfig:
# Unsloth-овский UnslothSFTTrainer.py (compiled cache) пересобирает SFTConfig
# с явным белым списком аргументов. На group_by_length мы уже натыкались на
# TypeError. Trainer читает self.args.packing в момент _prepare_dataset,
# поэтому setattr ДО создания SFTTrainer срабатывает гарантированно.
training_args.packing = True
training_args.eval_packing = False   # eval per-пример: eval_loss сравним по сэмплам с baseline_masked

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=ds_split["train"],
    eval_dataset=eval_subset,
)

# Маскируем prompt-токены: labels = -100 для всего, кроме ответа ассистента.
# С packing в одной упакованной последовательности будет несколько пар маркеров
# user/assistant — train_on_responses_only в свежих версиях Unsloth (см.
# discussion #2828) корректно обрабатывает все пары.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

### Sanity check: packing + маскирование

Проверяем три вещи:
1. **Packing работает**: `seq_length` в батче должна быть близка к `MAX_SEQ_LEN`, а **не** ~600 (средняя длина одного примера). Если seq_length маленький — packing не включился.
2. **Несколько assistant-сегментов в одной упакованной последовательности**: считаем число «контигуальных групп» токенов с `label != -100`. Если packing работает + маска работает, ожидаем 3–5 сегментов в каждой packed-строке.
3. **Trained-токены — это и правда ответы ассистента**: декодим их и читаем глазами. Trained-текст — конкатенация всех ответов в упакованной последовательности, разделённых `<|im_end|>`.

**Когда стоит начать переживать:**
- `seq_length` в батче ≈ 200–600 → **packing не сработал**, проверь `training_args.packing = True` и версию Unsloth (нужна свежая, см. PR #3566).
- `aggregate_ratio < 1%` или `> 95%` — `train_on_responses_only` не пробил маркеры. Скорее всего, расхождение между ChatML-шаблоном в Cell 14 и маркерами в Cell 18 (`<|im_start|>user\n`). В этом случае посмотри декодинг начала первой последовательности — что именно там стоит вместо ожидаемого маркера.
- `assistant segments per packed sequence == 1` — несмотря на packing, маркер ищется только по первой паре. Нужна более новая версия Unsloth.
- В декодинге trained-токенов попадаются куски user-промптов или русский от лица user-а — chat template неправильный.

In [ ]:
batch = next(iter(trainer.get_train_dataloader()))
input_ids_batch = batch["input_ids"]
labels_batch    = batch["labels"]
B, L = input_ids_batch.shape
print(f"Batch: {B} packed-последовательностей × {L} токенов")
print(f"padding_side: {tokenizer.padding_side}")

# 1) Packing работает? seq_length должна быть близка к MAX_SEQ_LEN.
print(f"\n--- (1) Packing check ---")
print(f"L = {L}, MAX_SEQ_LEN = {MAX_SEQ_LEN}")
if L < MAX_SEQ_LEN * 0.7:
    print(f"WARNING: L = {L} сильно меньше MAX_SEQ_LEN — packing, похоже, НЕ сработал.")
    print("        Проверь training_args.packing = True и что Unsloth поддерживает packing.")
else:
    print(f"OK: L близок к MAX_SEQ_LEN — packing работает.")

# 2) Сколько assistant-сегментов в каждой последовательности.
#    Считаем как число «контигуальных групп токенов с label != -100».
print(f"\n--- (2) Assistant segments per packed sequence ---")
for i in range(B):
    labels = labels_batch[i]
    mask   = labels != -100
    # Число «переходов» 0→1 в маске = число assistant-сегментов.
    transitions = ((mask[1:].int() - mask[:-1].int()) == 1).sum().item()
    if mask[0]:
        transitions += 1
    print(f"  [{i}] assistant segments: {transitions}")

# 3) Per-sequence статистика + декодинг.
print(f"\n--- (3) Per-sequence ratio + decoded trained tokens ---")
for i in range(B):
    input_ids = input_ids_batch[i]
    labels    = labels_batch[i]
    mask      = labels != -100
    n_masked  = (~mask).sum().item()
    n_trained = mask.sum().item()
    ratio_i   = n_trained / L

    trained_text = tokenizer.decode(input_ids[mask], skip_special_tokens=False)
    print(f"  [{i}] masked={n_masked:5d}  trained={n_trained:5d}  ratio={ratio_i:6.2%}")
    print(f"      trained → {trained_text[:160]!r}")

# 4) Агрегированный ratio по всему батчу.
total_trained = (labels_batch != -100).sum().item()
total_tokens  = labels_batch.numel()
agg_ratio     = total_trained / total_tokens
print(f"\n--- (4) Aggregate ---")
print(f"trained_tokens = {total_trained}/{total_tokens} → ratio = {agg_ratio:.2%}")

if agg_ratio < 0.01:
    print("WARNING: маска НЕ сработала — train_on_responses_only не пробил маркер.")
    print("Декодинг начала первой последовательности (для отладки):")
    print(repr(tokenizer.decode(input_ids_batch[0][:120])))
    print("Сравни с ожидаемым '<|im_start|>user\\n...'")
elif agg_ratio > 0.95:
    print("WARNING: маска НЕ сработала — почти всё попадает в loss.")
    print("Маркер instruction_part='<|im_start|>user\\n' видимо не нашёлся.")
else:
    print("OK: маскирование работает корректно.")

In [ ]:
import os

ckpt_dirs = []
if os.path.isdir(OUTPUT_DIR):
    ckpt_dirs = sorted(d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-"))

if ckpt_dirs:
    print(f"Найдено {len(ckpt_dirs)} чекпоинт(ов): {ckpt_dirs}")
    print(f"Продолжаем с последнего: {ckpt_dirs[-1]}")
    trainer.train(resume_from_checkpoint=True)
else:
    print("Чекпоинтов нет — обучение с нуля.")
    trainer.train()

In [ ]:
with open(LOG_PATH, "w", encoding="utf-8") as f:
    json.dump(trainer.state.log_history, f, indent=2, ensure_ascii=False)

trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Адаптер: {ADAPTER_DIR}")
print(f"Логи:    {LOG_PATH}")
print(f"Шагов сделано: {trainer.state.global_step}")

## Шаг 2.2 — Оценка качества (perplexity)

Загружаем **сохранённый адаптер**, а не свежую модель с нулевыми LoRA-весами.

После классической PPL по полному `text` считаются **assistant-only** PPL на собственном val и на **общем val**. Common val теперь внешний: `shuffle(seed=SEED)[0:COMMON_VAL_HOLDOUT_SIZE]`, а baseline train pool начинается с `COMMON_VAL_HOLDOUT_SIZE`. Поэтому финальная метрика не пересекается с train/selection pools. В `final_metrics.json` пишется ключ **`perplexity_assistant_only`** (= common val), чтобы стратегии подтягивали baseline без константы-фолбэка.

Загрузку делаем через `FastLanguageModel.from_pretrained(ADAPTER_DIR, ...)`: Unsloth увидит `adapter_config.json`, сам подтянет базовую модель в 4-bit и навесит LoRA. Использовать «голый» `AutoModelForCausalLM.from_pretrained` нельзя — после `import unsloth` (ячейка 6) в `transformers` глобально пропатчен `Qwen2Attention.forward`, который ждёт атрибут `apply_qkv`.

`FastLanguageModel.for_inference(model)` дополнительно ускоряет инференс (отключает gradient checkpointing, переключает в eval).

Метрика — **perplexity по полному тексту примера**. В этом ноутбуке `text` уже отрендерен на этапе подготовки данных (Cell 14) через явно зафиксированный Qwen2.5 ChatML-шаблон. Для сравнимости другие стратегии должны использовать тот же рендер и тот же common val.

> Перед загрузкой освобождаем VRAM от обучающей копии модели/тренера через `del trainer, model` + `torch.cuda.empty_cache()`.

In [ ]:
import json
import math, time, gc
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from transformers import AutoTokenizer

# Освобождаем VRAM от обучающей копии модели/тренера
del trainer, model
gc.collect()
torch.cuda.empty_cache()


def load_trained_model():
    model, tok = FastLanguageModel.from_pretrained(
        model_name=ADAPTER_DIR,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    return model, tok


def compute_perplexity(model, tok, texts):
    losses = []
    for text in texts:
        inputs = tok(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SEQ_LEN,
        ).to(model.device)
        with torch.no_grad():
            loss = model(**inputs, labels=inputs["input_ids"]).loss
        losses.append(loss.item())
    return math.exp(sum(losses) / len(losses))


def compute_assistant_only_perplexity(model, tok, examples, max_length=MAX_SEQ_LEN):
    """PPL только на assistant-токенах (тот же код, что selection_entropy Cell 24)."""
    user_marker_ids = tok("<|im_start|>user\n", add_special_tokens=False)["input_ids"]
    asst_marker_ids = tok("<|im_start|>assistant\n", add_special_tokens=False)["input_ids"]

    total_loss = 0.0
    total_tokens = 0
    skipped = 0

    for ex in examples:
        if "text" in ex:
            text = ex["text"]
        elif "messages" in ex:
            text = tok.apply_chat_template(
                ex["messages"], tokenize=False, add_generation_prompt=False,
            )
        else:
            raise ValueError(f"Жду 'text' или 'messages', есть: {list(ex.keys())}")

        inputs = tok(text, return_tensors="pt", truncation=True, max_length=max_length).to(model.device)
        input_ids = inputs["input_ids"][0]

        labels = input_ids.clone()
        labels[:] = -100
        ids_list = input_ids.tolist()
        i = 0
        while i < len(ids_list):
            if ids_list[i : i + len(asst_marker_ids)] == asst_marker_ids:
                start = i + len(asst_marker_ids)
                end = len(ids_list)
                for j in range(start, len(ids_list) - len(user_marker_ids) + 1):
                    if ids_list[j : j + len(user_marker_ids)] == user_marker_ids:
                        end = j
                        break
                labels[start:end] = input_ids[start:end]
                i = end
            else:
                i += 1

        n_assistant = (labels != -100).sum().item()
        if n_assistant == 0:
            skipped += 1
            continue

        with torch.no_grad():
            out = model(input_ids=input_ids.unsqueeze(0), labels=labels.unsqueeze(0))

        total_loss += out.loss.item() * n_assistant
        total_tokens += n_assistant

    if total_tokens == 0:
        raise RuntimeError("Не нашлось ни одного assistant-токена. Проверь маркеры/chat template.")
    if skipped:
        print(f"  WARNING: пропущено {skipped} примеров без assistant-сегментов (truncation?)")
    print(f"  Усреднено по {total_tokens} assistant-токенам, {len(examples) - skipped} примеров")
    return math.exp(total_loss / total_tokens)


eval_model, eval_tok = load_trained_model()

n_eval = min(FINAL_EVAL_SAMPLES, len(ds_split["test"]))
# text уже отрендерен в Cell 14 через явно зафиксированный Qwen2.5 ChatML-шаблон.
texts = [ex["text"] for ex in ds_split["test"].select(range(n_eval))]

t0 = time.time()
ppl = compute_perplexity(eval_model, eval_tok, texts)
elapsed = time.time() - t0

# --- Assistant-only PPL (как selection_entropy / selection_loss_rho Cell 24) ---
n_asst = min(FINAL_EVAL_SAMPLES, len(ds_split["test"]))
own_val = ds_split["test"].select(range(n_asst))
print(f"\n--- PPL на СОБСТВЕННОМ val ({len(own_val)} примеров, assistant-only) ---")
ppl_asst_own = compute_assistant_only_perplexity(eval_model, eval_tok, own_val)
print(f"  → {ppl_asst_own:.4f}")

print(
    f"\nПостроение ОБЩЕГО val (reserved {COMMON_VAL_HOLDOUT_SIZE:,}, seed=SEED; "
    "исключён из train/selection pools)..."
)
stock_tok = get_chat_template(
    AutoTokenizer.from_pretrained(MODEL_NAME),
    chat_template="qwen-2.5",
)
ds_full_m = load_dataset(DATASET_NAME, split="train")
ds_common = ds_full_m.shuffle(seed=SEED).select(range(COMMON_VAL_HOLDOUT_SIZE))


def _render_common(examples):
    texts = [
        stock_tok.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in examples["conversations"]
    ]
    return {"text": texts}


ds_common = ds_common.map(
    _render_common,
    batched=True,
    remove_columns=ds_common.column_names,
    desc="render common val",
)
common_val_size = min(FINAL_EVAL_SAMPLES, len(ds_common))
common_val = ds_common.select(range(common_val_size))

print(f"\n--- PPL на ОБЩЕМ val ({len(common_val)} примеров, assistant-only, главная метрика) ---")
ppl_asst_common = compute_assistant_only_perplexity(eval_model, eval_tok, common_val)
print(f"  → {ppl_asst_common:.4f}")

metrics = {
    "perplexity":        round(ppl, 4),
    "eval_time_sec":     round(elapsed, 2),
    "samples_evaluated": n_eval,
    "adapter":           ADAPTER_DIR,
    "model":             MODEL_NAME,
    "subsample_size":    SUBSAMPLE_SIZE,
    "training_config":   f"packing=True, train_on_responses_only, subsample={SUBSAMPLE_SIZE}, epochs={NUM_EPOCHS}, warmup_ratio={WARMUP_RATIO}",
    "perplexity_assistant_only": round(ppl_asst_common, 4),
    "perplexity_assistant_only_on_own_val": round(ppl_asst_own, 4),
    "perplexity_assistant_only_on_common_val": round(ppl_asst_common, 4),
    "comparison_to_baseline": {
        "note": "Сам baseline: common val заранее зарезервирован и исключён из train/selection pools.",
        "val_set": f"reserved common val (shuffle(seed=SEED)[0:{COMMON_VAL_HOLDOUT_SIZE}], first {len(common_val)} examples)",
        "baseline_strategy": f"random_{SUBSAMPLE_SIZE} (baseline_masked_packed_base_90k_1ep_noleak)",
        "baseline_ppl_asst": round(ppl_asst_common, 4),
        "baseline_source": "this run (baseline_masked_packed_base_90k_1ep_noleak)",
        "strategy_ppl_asst": round(ppl_asst_common, 4),
        "delta_absolute": 0.0,
        "delta_percent": 0.0,
        "verdict": "baseline_masked_packed_base_90k_1ep_noleak (reference)",
        "noise_floor": 0.03,
        "samples_evaluated": len(common_val),
    },
}

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f"\nPerplexity (full text): {ppl:.4f}")
print(f"Время (full text):      {elapsed:.1f} сек на {n_eval} примерах")
print(f"Метрики:                {METRICS_PATH}")

## (Опционально) Сохранение результатов между сессиями

Сессия Colab умрёт через несколько часов — всё в `/content/` пропадёт. Рекомендуемая схема:

| Что | Куда |
|---|---|
| LoRA-адаптер (~30 МБ) | HuggingFace Hub (приватный репо) |
| `training_log.json`, `final_metrics.json` | Скачать локально и закоммитить в git |

Альтернатива — смонтировать Google Drive и скопировать туда `outputs/baseline_masked_packed/` целиком. Если у тебя нет HF-аккаунта или не хочется заводить токен — следующие две ячейки можно пропустить.

Токен берём через `getpass`, чтобы он не попал в git.

In [ ]:
# Раскомментируй и впиши свой username + желаемое имя репо
# HF_REPO_ID = "your-username/qwen2.5-1.5b-ru-baseline-masked-packed-base-90k-1ep"
HF_REPO_ID = None

if HF_REPO_ID is not None:
    from huggingface_hub import HfApi, login
    from getpass import getpass

    login(token=getpass("HF token (write scope): "))

    api = HfApi()
    api.create_repo(HF_REPO_ID, private=True, exist_ok=True)
    api.upload_folder(
        folder_path=ADAPTER_DIR,
        repo_id=HF_REPO_ID,
        commit_message="baseline_masked_packed adapter (Qwen2.5-1.5B base, 90k random subsample, 1 epoch, packing=True, train_on_responses_only)",
    )
    api.upload_file(
        path_or_fileobj=METRICS_PATH,
        path_in_repo="final_metrics.json",
        repo_id=HF_REPO_ID,
    )
    api.upload_file(
        path_or_fileobj=LOG_PATH,
        path_in_repo="training_log.json",
        repo_id=HF_REPO_ID,
    )
    print(f"Адаптер и логи запушены в https://huggingface.co/{HF_REPO_ID}")
else:
    print("HF_REPO_ID не задан — пропускаем пуш на Hub.")
    print("Перед закрытием Colab скачай:")
    print(f"  - {ADAPTER_DIR}/")
    print(f"  - {LOG_PATH}")
    print(f"  - {METRICS_PATH}")